# Spark Setup

In [ ]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pymongo import MongoClient
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp, expr, unix_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

HOST_IP = "192.168.64.1"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .getOrCreate()
)

# Schema setup for easy handling

In [ ]:
# Reusable schema matching producer payload
EVENT_SCHEMA = StructType([
    StructField("event_id", StringType(), True),
    StructField("batch_id", IntegerType(), True),
    StructField("car_plate", StringType(), True),
    StructField("camera_id", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("speed_reading", DoubleType(), True),
    StructField("producer_id", StringType(), True),
    StructField("schema_version", StringType(), True)
])

def read_camera_stream(topic_name, watermark_minutes=10):
    """Read Kafka topic and parse into structured streaming DataFrame"""
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "localhost:9092")
        .option("subscribe", topic_name)
        .option("startingOffsets", "latest")  # Or "earliest" for testing
        .option("failOnDataLoss", "false")
        .load()
        .selectExpr("CAST(value AS STRING) AS json_value")
        .select(from_json(col("json_value"), EVENT_SCHEMA).alias("data"))
        .select("data.*")
        .withColumn("event_time", to_timestamp(col("timestamp")))
        .withWatermark("event_time", f"{watermark_minutes} minutes")  # Critical for state management
    )

# Create streams for all three cameras
stream_a = read_camera_stream("camera-events-A")
stream_b = read_camera_stream("camera-events-B")
stream_c = read_camera_stream("camera-events-C")
print("✓ Kafka streams created with watermarks")

In [ ]:
# Load speed limits from MongoDB or camera.csv
camera_limits = {row['camera_id']: row['speed_limit'] for _, row in pd.read_csv('data/camera.csv').iterrows()}
speed_limits_broadcast = spark.sparkContext.broadcast(camera_limits)

# UDF for instantaneous violation check
def check_instantaneous_violation(camera_id, speed):
    limit = speed_limits_broadcast.value.get(camera_id, float('inf'))
    return speed > limit

from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

is_speeding_udf = udf(check_instantaneous_violation, BooleanType())